In [ ]:
# Cell 1 - UTCI extreme heat: user inputs and run controls

from pathlib import Path

# Required user inputs
ISO3 = "HTI"
AS_OF_DATE = "2025-12-31"  # YYYY-MM-DD

# Extreme heat definition
K_CONSEC_DAYS = 3
ABS_THRESHOLDS_C = [32.0, 38.0, 46.0]
REL_THRESHOLD_Q = 0.95
USE_RELATIVE_THRESHOLD = False  # default: absolute thresholds only

# Baseline and analysis windows
BASELINE_START_YEAR = 1991
BASELINE_END_YEAR = 2020
RECENT_MONTHS = 12

# CDS request buffer in degrees around admin bounds
CDS_BUFFER_DEG = 0.25

# Input data sources
COD_GDB_ZIP = Path("./data/cod-ab/global_admin_boundaries_matched_latest.gdb.zip")
COD_ADMIN2_LAYER = "admin2"
worldpop_name = f"{ISO3.lower()}_pop_2025_CN_100m_R2025A_v1.tif"
WORLDPOP_PATH = Path(f"./data/population/{worldpop_name}")

# Output root and processing options
OUT_ROOT = Path("./outputs")
XR_CHUNKS = {"time": 60, "latitude": 200, "longitude": 200}

assert len(ISO3) == 3, "ISO3 must be a 3-letter code."

In [ ]:
# Cell 2 - Imports and lightweight utilities

import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray  # noqa: F401

from shapely.geometry import box
from shapely.ops import unary_union

import rasterio
from rasterio.enums import Resampling

from wia_pipelines.config import RunConfig, initialize_run_metadata, validate_run_metadata

# Optional zonal stats (preferred when available)
try:
    HAS_RASTERSTATS = True
except Exception:
    HAS_RASTERSTATS = False

warnings.filterwarnings("ignore", category=UserWarning)


def iso_date(s: str) -> str:
    """Validate YYYY-MM-DD and return normalized string."""
    datetime.strptime(s, "%Y-%m-%d")
    return s


AS_OF_DATE = iso_date(AS_OF_DATE)


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

In [ ]:
# Cell 3 - Output directories and schema-compliant run metadata scaffold

# Derive analysis window from as-of date
as_of = pd.Timestamp(AS_OF_DATE)
end_date = as_of.normalize()
start_date = (end_date - pd.DateOffset(months=RECENT_MONTHS)).normalize()

# Build schema-compliant run metadata and base layout
RUN_CONFIG = RunConfig(
    hazard="heat",
    iso3=ISO3,
    as_of_date=AS_OF_DATE,
    lookback_months=RECENT_MONTHS,
    output_root=OUT_ROOT,
    target_adm_level=2,
    buffer_km=0.0,
)

run_id = RUN_CONFIG.run_id
BASE_DIR = ensure_dir(OUT_ROOT / "heat" / ISO3 / run_id)

# Canonical run contract directories (shared across hazards)
DIRS = {
    "base": BASE_DIR,
    "raw": ensure_dir(BASE_DIR / "raw"),
    "intermediate": ensure_dir(BASE_DIR / "intermediate"),
    "rasters": ensure_dir(BASE_DIR / "rasters"),
    "tables": ensure_dir(BASE_DIR / "tables"),
    "qc": ensure_dir(BASE_DIR / "qc"),
    "logs": ensure_dir(BASE_DIR / "logs"),
    # Backward-compatible aliases used in existing UTCI cells
    "baseline_raw": ensure_dir(BASE_DIR / "raw" / "baseline"),
    "recent_raw": ensure_dir(BASE_DIR / "raw" / "recent"),
    "baseline_ref": ensure_dir(BASE_DIR / "intermediate" / "baseline_ref"),
    "masks_native": ensure_dir(BASE_DIR / "rasters" / "masks_native"),
    "masks_worldpop": ensure_dir(BASE_DIR / "rasters" / "masks_worldpop"),
    "pop_exposed": ensure_dir(BASE_DIR / "rasters" / "pop_exposed"),
}

# Shared per-country cache directories
COUNTRY_CACHE_DIR = ensure_dir(OUT_ROOT / "heat" / ISO3 / "_cache")
CACHE_DIRS = {
    "baseline_raw": ensure_dir(COUNTRY_CACHE_DIR / "baseline_raw"),
    "baseline_ref": ensure_dir(COUNTRY_CACHE_DIR / "baseline_ref"),
}

# Manifest used for artifact-level traceability
MANIFEST_PATH = DIRS["logs"] / "artifact_manifest.csv"


def record_artifact(kind: str, path: Path, notes: str = "") -> None:
    row = {
        "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "kind": kind,
        "path": str(path),
        "notes": notes,
    }
    df = pd.DataFrame([row])
    if MANIFEST_PATH.exists():
        df.to_csv(MANIFEST_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(MANIFEST_PATH, mode="w", header=True, index=False)


run_metadata = initialize_run_metadata(
    RUN_CONFIG,
    paths={
        "base": DIRS["base"],
        "raw": DIRS["raw"],
        "rasters": DIRS["rasters"],
        "tables": DIRS["tables"],
        "qc": DIRS["qc"],
        "logs": DIRS["logs"],
        "cache": COUNTRY_CACHE_DIR,
    },
)

run_metadata["pipeline"] = "extreme_heat_utci"
run_metadata["utci_config"] = {
    "recent_window": {
        "months": RECENT_MONTHS,
        "start_date": str(start_date.date()),
        "end_date": str(end_date.date()),
        "k_consecutive_days": K_CONSEC_DAYS,
    },
    "thresholds": {
        "absolute_c": ABS_THRESHOLDS_C,
        "relative_quantile": REL_THRESHOLD_Q,
    },
    "baseline_years": {
        "start_year": BASELINE_START_YEAR,
        "end_year": BASELINE_END_YEAR,
    },
    "inputs": {
        "cod_gdb_zip": str(COD_GDB_ZIP),
        "cod_layer": COD_ADMIN2_LAYER,
        "worldpop_path": str(WORLDPOP_PATH),
    },
    "cds_buffer_deg": CDS_BUFFER_DEG,
}
run_metadata["cache_dirs"] = {k: str(v) for k, v in CACHE_DIRS.items()}

RUN_METADATA_PATH = DIRS["base"] / "run_metadata.json"


def update_run_metadata_artifact(kind: str, path: Path, notes: str = "") -> None:
    run_metadata.setdefault("artifacts", []).append({"kind": kind, "path": str(path), "notes": notes})
    record_artifact(kind, path, notes)


validate_run_metadata(run_metadata)
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))
update_run_metadata_artifact("metadata_init", RUN_METADATA_PATH, "Initial schema-compliant run metadata")

print("Run workspace:", BASE_DIR)
print("Window:", start_date.date(), "->", end_date.date())

In [ ]:
# Cell 4 — Load admin2 boundaries (single source of truth)


def load_admin2_for_iso3(
    iso3: str, gdb_zip: Path = COD_GDB_ZIP, layer: str = COD_ADMIN2_LAYER
) -> gpd.GeoDataFrame:
    iso3 = iso3.upper().strip()

    if not gdb_zip.exists():
        raise FileNotFoundError(f"Admin boundaries file not found: {gdb_zip.resolve()}")

    vsizip_path = f"/vsizip/{gdb_zip.as_posix()}"
    gdf = gpd.read_file(vsizip_path, layer=layer)

    # Required fields
    required = {"iso3", "geometry"}
    missing = required - set(gdf.columns)
    if missing:
        raise ValueError(f"Missing required column(s) in {layer}: {sorted(missing)}")

    gdf_iso = gdf.loc[gdf["iso3"] == iso3].copy()
    if gdf_iso.empty:
        raise ValueError(f"No admin2 features found for ISO3={iso3} in layer '{layer}'")

    # Normalise ADM2 PCODE field name to 'adm2_pcode'
    pcode_candidates = ["adm2_pcode", "ADM2_PCODE", "ADM2PCODE", "pcode", "PCode"]
    found = None
    for c in pcode_candidates:
        if c in gdf_iso.columns:
            found = c
            break
    if not found:
        raise ValueError("No ADM2 pcode field found. Please confirm the admin2 PCODE column name in the GDB.")
    if found != "adm2_pcode":
        gdf_iso = gdf_iso.rename(columns={found: "adm2_pcode"})

    # Drop null/empty geometries + attempt validity fix
    gdf_iso = gdf_iso[gdf_iso.geometry.notnull()].copy()
    try:
        gdf_iso["geometry"] = gdf_iso.geometry.make_valid()
    except Exception:
        gdf_iso["geometry"] = gdf_iso.geometry.buffer(0)

    gdf_iso = gdf_iso[~gdf_iso.geometry.is_empty].copy()

    # Basic sanity checks
    if gdf_iso.crs is None:
        raise ValueError("Admin2 layer CRS is missing; cannot proceed safely.")

    # Ensure unique pcodes (or at least detect duplicates)
    dupes = gdf_iso["adm2_pcode"].duplicated().sum()
    if dupes:
        print(f"Warning: {dupes} duplicate adm2_pcode values detected (will still proceed).")

    return gdf_iso


admin2_gdf = load_admin2_for_iso3(ISO3)

print(f"Loaded admin2 features: {len(admin2_gdf)} for {ISO3}")
print("CRS:", admin2_gdf.crs)
print(
    "Columns (subset):",
    [c for c in ["iso3", "adm2_pcode", "adm2_name", "name", "geometry"] if c in admin2_gdf.columns],
)

# Record
update_run_metadata_artifact(
    "admin2_gdf_loaded",
    DIRS["logs"] / "admin2_loaded.txt",
    f"{len(admin2_gdf)} features; CRS={admin2_gdf.crs}",
)
(DIRS["logs"] / "admin2_loaded.txt").write_text(
    f"ISO3={ISO3}\ncount={len(admin2_gdf)}\ncrs={admin2_gdf.crs}\n"
)

In [ ]:
# Cell 5 - Country AOI and buffered CDS request area

# Reproject admin2 to EPSG:4326 for AOI and CDS compatibility
admin2_4326 = admin2_gdf.to_crs("EPSG:4326")

# Union admin geometries without geometric buffer distortion
country_geom_4326 = unary_union(admin2_4326.geometry.values)
if country_geom_4326.is_empty:
    raise ValueError("Unified country geometry is empty after union.")

# Unbuffered admin bounds (W, S, E, N)
minx, miny, maxx, maxy = country_geom_4326.bounds
country_bounds_4326 = (float(minx), float(miny), float(maxx), float(maxy))

# CDS area with explicit coarse-grid buffer: [North, West, South, East]
west_cds = max(-180.0, country_bounds_4326[0] - CDS_BUFFER_DEG)
south_cds = max(-90.0, country_bounds_4326[1] - CDS_BUFFER_DEG)
east_cds = min(180.0, country_bounds_4326[2] + CDS_BUFFER_DEG)
north_cds = min(90.0, country_bounds_4326[3] + CDS_BUFFER_DEG)
cds_bbox = [north_cds, west_cds, south_cds, east_cds]

print("Country bounds (EPSG:4326):")
print(
    f"  West={country_bounds_4326[0]:.4f}, South={country_bounds_4326[1]:.4f}, "
    f"East={country_bounds_4326[2]:.4f}, North={country_bounds_4326[3]:.4f}"
)
print("CDS buffer (deg):", CDS_BUFFER_DEG)
print("CDS area (NWSE):", cds_bbox)

import hashlib

bounds_payload = {
    "iso3": ISO3,
    "bounds_wsen": [float(x) for x in country_bounds_4326],
    "crs": "EPSG:4326",
}

bounds_json = json.dumps(bounds_payload, sort_keys=True)
bounds_hash = hashlib.sha256(bounds_json.encode("utf-8")).hexdigest()[:12]
BOUNDS_HASH_PATH = COUNTRY_CACHE_DIR / "aoi_bounds_hash.json"

bounds_record = {
    "iso3": ISO3,
    "bounds_wsen": bounds_payload["bounds_wsen"],
    "crs": "EPSG:4326",
    "bounds_hash": bounds_hash,
    "source": "admin2 union from COD global_admin_boundaries_matched_latest.gdb.zip",
}
BOUNDS_HASH_PATH.write_text(json.dumps(bounds_record, indent=2))

run_metadata.setdefault("aoi", {})
run_metadata["aoi"].update(
    {
        "bounds_wsen": bounds_payload["bounds_wsen"],
        "bounds_hash": bounds_hash,
        "cds_area": cds_bbox,
        "cds_buffer_deg": CDS_BUFFER_DEG,
    }
)
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

AOI_GEOJSON_PATH = DIRS["qc"] / f"{ISO3}_admin2_union_aoi.geojson"
gpd.GeoSeries([country_geom_4326], crs="EPSG:4326").to_file(AOI_GEOJSON_PATH, driver="GeoJSON")
update_run_metadata_artifact("aoi_geometry", AOI_GEOJSON_PATH, "Unified admin2 geometry")

bbox_area_deg2 = (country_bounds_4326[2] - country_bounds_4326[0]) * (
    country_bounds_4326[3] - country_bounds_4326[1]
)
aoi_area_deg2 = country_geom_4326.area
print(f"AOI area (deg^2): {aoi_area_deg2:.2f}")
print(f"BBox area (deg^2): {bbox_area_deg2:.2f}")

In [ ]:
# Cell 6 — Load WorldPop raster (explicit path)

if WORLDPOP_PATH is None:
    raise ValueError("WORLDPOP_PATH must be set explicitly (manual per-country download).")

if not WORLDPOP_PATH.exists():
    raise FileNotFoundError(f"WorldPop raster not found: {WORLDPOP_PATH.resolve()}")

print("WorldPop raster:", WORLDPOP_PATH)

worldpop = rioxarray.open_rasterio(WORLDPOP_PATH, masked=True).squeeze(drop=True)

if worldpop.rio.crs is None:
    raise ValueError("WorldPop raster CRS is missing; cannot proceed safely.")

print("WorldPop CRS:", worldpop.rio.crs)
print("WorldPop shape:", tuple(worldpop.shape))
print("WorldPop resolution:", worldpop.rio.resolution())

# QC: ensure overlap with AOI bbox (reproject AOI bbox to raster CRS)
from shapely.geometry import box

aoi_bbox_4326 = box(minx, miny, maxx, maxy)  # from Cell 5
aoi_bbox_gdf = gpd.GeoDataFrame({"geometry": [aoi_bbox_4326]}, crs="EPSG:4326").to_crs(worldpop.rio.crs)
aoi_bbox_raster_crs = aoi_bbox_gdf.geometry.iloc[0]

rb = worldpop.rio.bounds()
raster_bbox_poly = box(rb[0], rb[1], rb[2], rb[3])

overlaps = raster_bbox_poly.intersects(aoi_bbox_raster_crs)
print("AOI bbox overlaps WorldPop raster bounds:", overlaps)
if not overlaps:
    raise ValueError("WorldPop raster does not overlap AOI bbox. Check ISO3/raster selection.")

# Record
update_run_metadata_artifact("worldpop_raster", WORLDPOP_PATH, "WorldPop 100m pop counts (manually provided)")
run_metadata.setdefault("utci_config", {}).setdefault("inputs", {})["worldpop_path"] = str(WORLDPOP_PATH)
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 7 - Preflight coverage checks before heavy CDS downloads

from wia_pipelines.hazards.coverage_checks import (
    check_worldpop_coverage,
    plot_grid_overlay_figure,
    plot_raster_overlay_figure,
    run_cds_single_month_check,
    utci_sample_request,
)
from wia_pipelines.core.worldpop import bbox_coverage_report

# Use first month in the analysis window for a lightweight sample check
SAMPLE_YEAR = int(start_date.year)
SAMPLE_MONTH = int(start_date.month)

admin_bounds_wsen = country_bounds_4326

# 1) WorldPop coverage check
wp_cov = check_worldpop_coverage(admin_bounds_wsen, WORLDPOP_PATH)
print("WorldPop coverage %:", round(wp_cov["coverage_pct"], 3), "full=", wp_cov["full_coverage"])

# 2) UTCI sample coverage check from CDS
preflight_dir = ensure_dir(DIRS["logs"] / "preflight")
sample_zip = preflight_dir / f"{ISO3}_utci_sample_{SAMPLE_YEAR}{SAMPLE_MONTH:02d}.zip"
sample_req = utci_sample_request(SAMPLE_YEAR, SAMPLE_MONTH, cds_bbox)

sample = run_cds_single_month_check(
    dataset="derived-utci-historical",
    request=sample_req,
    output_zip=sample_zip,
)
if not sample["ok"]:
    raise RuntimeError(f"UTCI preflight sample download failed: {sample['error']}")
if sample.get("sample_bounds_4326") is None:
    raise RuntimeError("UTCI preflight sample has no inferred bounds.")

utci_cov = bbox_coverage_report(admin_bounds_wsen, tuple(sample["sample_bounds_4326"]))
print("UTCI sample coverage %:", round(utci_cov["coverage_pct"], 3), "full=", utci_cov["full_coverage"])

# 3) Visual QA overlays
PRE_QC_DIR = ensure_dir(DIRS["qc"] / "preflight")
utci_nc = Path(sample["first_nc_path"])

wp_raster_fig = PRE_QC_DIR / f"{ISO3}_preflight_worldpop_raster_overlay.png"
wp_grid_fig = PRE_QC_DIR / f"{ISO3}_preflight_worldpop_grid_overlay.png"
utci_raster_fig = PRE_QC_DIR / f"{ISO3}_preflight_utci_raster_overlay.png"
utci_grid_fig = PRE_QC_DIR / f"{ISO3}_preflight_utci_grid_overlay.png"

plot_raster_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    WORLDPOP_PATH,
    "raster",
    wp_raster_fig,
    f"{ISO3} Preflight WorldPop Raster + Admin + AOI",
)
plot_grid_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    WORLDPOP_PATH,
    "raster",
    wp_grid_fig,
    f"{ISO3} Preflight WorldPop Grid + Admin + AOI",
)
plot_raster_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    utci_nc,
    "netcdf",
    utci_raster_fig,
    f"{ISO3} Preflight UTCI Sample Raster + Admin + AOI",
)
plot_grid_overlay_figure(
    admin2_gdf,
    admin_bounds_wsen,
    utci_nc,
    "netcdf",
    utci_grid_fig,
    f"{ISO3} Preflight UTCI Sample Grid + Admin + AOI",
)

run_metadata["preflight_coverage"] = {
    "sample_year": SAMPLE_YEAR,
    "sample_month": SAMPLE_MONTH,
    "worldpop": wp_cov,
    "utci_sample": {
        **sample,
        "coverage_pct": float(utci_cov["coverage_pct"]),
        "full_coverage": bool(utci_cov["full_coverage"]),
    },
    "figures": {
        "worldpop_raster_overlay": str(wp_raster_fig),
        "worldpop_grid_overlay": str(wp_grid_fig),
        "utci_raster_overlay": str(utci_raster_fig),
        "utci_grid_overlay": str(utci_grid_fig),
    },
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

if not utci_cov["full_coverage"]:
    raise RuntimeError(
        f"UTCI preflight coverage is not full ({utci_cov['coverage_pct']:.2f}%). "
        "Increase CDS buffer and rerun preflight."
    )

print("Preflight coverage checks passed.")

In [ ]:
# Cell 8 — CDS downloads (recent by default; baseline optional for relative threshold)
# Refactored to match CDS enumerations for derived-utci-historical:
#   - product_type: consolidated_dataset / intermediate_dataset
#   - variable: universal_thermal_climate_index_daily_statistics (smaller; includes daily max/min)
#
# Source dataset reference: https://cds.climate.copernicus.eu/datasets/derived-utci-historical

import zipfile
import calendar
import hashlib
import time
import cdsapi

# ---- Constants aligned to CDS dataset ----
DATASET_UTCI = "derived-utci-historical"

PRODUCT_CONSOLIDATED = "consolidated_dataset"
PRODUCT_INTERIM = "intermediate_dataset"

UTCI_DOWNLOAD_STATUS_RECENT = DIRS["logs"] / "utci_recent_download_status.json"
UTCI_DOWNLOAD_STATUS_BASELINE = DIRS["logs"] / "utci_baseline_download_status.json"


def _write_progress_status(
    path: Path,
    stage: str,
    processed: int,
    total: int,
    ok: int,
    failed: int,
    started_at: float,
    current: str | None = None,
):
    elapsed = time.perf_counter() - started_at
    rate = processed / elapsed if elapsed > 0 else 0.0
    remaining = max(0, total - processed)
    eta = remaining / rate if rate > 0 else None
    payload = {
        "stage": stage,
        "processed": int(processed),
        "total": int(total),
        "ok": int(ok),
        "failed": int(failed),
        "pct_complete": float((processed / total) * 100.0) if total else 100.0,
        "elapsed_seconds": float(round(elapsed, 2)),
        "rate_items_per_second": float(round(rate, 4)),
        "eta_seconds": None if eta is None else float(round(eta, 2)),
        "current": current,
    }
    path.write_text(json.dumps(payload, indent=2))


# ---- AOI bounds hash helper (baseline cache guard) ----
# Requires: ISO3, country_bounds_4326, bounds_hash, COUNTRY_CACHE_DIR defined earlier (Cells 3/5)

BOUNDS_HASH_PATH = COUNTRY_CACHE_DIR / "aoi_bounds_hash.json"


def compute_bounds_hash(iso3: str, bounds_wsen: tuple[float, float, float, float]) -> tuple[str, dict]:
    """
    Compute a stable hash for the AOI bounds used to request CDS downloads.
    bounds_wsen: (W, S, E, N) in EPSG:4326
    """
    payload = {
        "iso3": iso3.upper(),
        "bounds_wsen": [float(x) for x in bounds_wsen],
        "crs": "EPSG:4326",
    }
    payload_json = json.dumps(payload, sort_keys=True)
    h = hashlib.sha256(payload_json.encode("utf-8")).hexdigest()[:12]
    return h, payload


def read_cached_bounds_hash(path: Path) -> str | None:
    if not path.exists():
        return None
    try:
        data = json.loads(path.read_text())
        return data.get("bounds_hash")
    except Exception:
        return None


def write_bounds_hash_file(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2))


# Compute current hash (based on current admin2 union bounds)
current_bounds_hash, _bounds_payload = compute_bounds_hash(ISO3, country_bounds_4326)

# Validate against cache, if present
cached_hash = read_cached_bounds_hash(BOUNDS_HASH_PATH)
if cached_hash and cached_hash != current_bounds_hash:
    raise RuntimeError(
        "Cached baseline AOI bounds hash does not match current admin2 union.\n"
        f"Cached:  {cached_hash}\n"
        f"Current: {current_bounds_hash}\n"
        "This likely means admin boundaries or AOI definition changed. "
        "Delete the country cache (OUT_ROOT/<ISO3>/_cache) to force a fresh baseline."
    )

# Ensure the bounds hash file exists (write-once behaviour)
if not BOUNDS_HASH_PATH.exists():
    bounds_record = {
        "iso3": ISO3,
        "bounds_wsen": _bounds_payload["bounds_wsen"],
        "crs": "EPSG:4326",
        "bounds_hash": current_bounds_hash,
        "source": "admin2 union from COD global_admin_boundaries_matched_latest.gdb.zip",
    }
    write_bounds_hash_file(BOUNDS_HASH_PATH, bounds_record)
    update_run_metadata_artifact(
        "aoi_bounds_hash",
        BOUNDS_HASH_PATH,
        "Country cache AOI bounds hash (baseline guard)",
    )


def months_all() -> list[str]:
    return [f"{m:02d}" for m in range(1, 13)]


def days_for_year_month(year: int, month: int) -> list[str]:
    """Leap-year aware day list for a given year-month."""
    last_day = calendar.monthrange(year, month)[1]
    return [f"{d:02d}" for d in range(1, last_day + 1)]


def cds_request(area, years, product_type=PRODUCT_CONSOLIDATED, months=None, days=None):
    """
    Create a CDS request dict for derived-utci-historical.
    area must be [North, West, South, East] in degrees (EPSG:4326).
    """
    if months is None:
        months = months_all()
    # NOTE: For safety, callers should pass month-specific valid days.
    # If days is None, we default to 01..31, which is only safe for Jan/Mar/May/Jul/Aug/Oct/Dec.
    if days is None:
        days = [f"{d:02d}" for d in range(1, 32)]

    return {
        "variable": ["universal_thermal_climate_index_daily_statistics"],
        "version": "1_1",
        "product_type": product_type,  # consolidated_dataset / intermediate_dataset
        "year": [str(y) for y in years],
        "month": months,
        "day": days,
        "area": area,
    }


def download_cds(dataset: str, request: dict, out_zip: Path) -> tuple[bool, str | None]:
    out_zip.parent.mkdir(parents=True, exist_ok=True)
    try:
        client = cdsapi.Client()
        result = client.retrieve(dataset, request)
        result.download(target=str(out_zip))
        return True, None
    except Exception as e:
        return False, str(e)


# ---- Baseline downloader (monthly; valid day lists) ----
def download_baseline_monthly(
    area,
    start_year: int,
    end_year: int,
    out_dir: Path,
    product_type: str = PRODUCT_CONSOLIDATED,
    status_path: Path | None = None,
):
    """
    Download baseline per year-month to avoid invalid month/day combinations.
    """
    man = []
    total = (end_year - start_year + 1) * 12
    ok_count = 0
    fail_count = 0
    started_at = time.perf_counter()
    if status_path is not None:
        _write_progress_status(status_path, "baseline_download", 0, total, ok_count, fail_count, started_at)

    step = 0
    for y in range(start_year, end_year + 1):
        for m in range(1, 13):
            step += 1
            days = days_for_year_month(y, m)
            req = cds_request(
                area,
                years=[y],
                product_type=product_type,
                months=[f"{m:02d}"],
                days=days,
            )
            out_zip = out_dir / f"baseline_{product_type}_{y}{m:02d}.zip"
            ok, err = download_cds(DATASET_UTCI, req, out_zip)
            man.append(
                {
                    "type": "baseline",
                    "product_type": product_type,
                    "year": y,
                    "month": f"{m:02d}",
                    "days": f"{days[0]}-{days[-1]}",
                    "ok": ok,
                    "error": err,
                    "path": str(out_zip),
                }
            )
            if ok:
                ok_count += 1
            else:
                fail_count += 1
            if status_path is not None:
                _write_progress_status(
                    status_path,
                    "baseline_download",
                    step,
                    total,
                    ok_count,
                    fail_count,
                    started_at,
                    current=f"{y}-{m:02d}",
                )
            if (step % 12 == 0) or (step == total):
                print(
                    f"Baseline download progress {step}/{total} ({(step / total) * 100:.1f}%) ok={ok_count} failed={fail_count}"
                )
    return man


# ---- Recent (last 12 months; monthly windows; consolidated + interim) ----
def months_for_last_12(as_of: str) -> list[tuple[int, int]]:
    end = pd.to_datetime(as_of).to_period("M")
    return [(int((end - n).year), int((end - n).month)) for n in range(0, 12)][::-1]


def _dates_last12(as_of: str) -> pd.DatetimeIndex:
    end = pd.to_datetime(as_of)
    start = (end - pd.DateOffset(months=12)) + pd.Timedelta(days=1)
    return pd.date_range(start=start, end=end, freq="D")


def _days_in_month_window(dates: pd.DatetimeIndex, year: int, month: int) -> list[str]:
    sel = dates[(dates.year == year) & (dates.month == month)]
    return sorted({f"{d:02d}" for d in sel.day})


def _zip_ok(path: Path) -> bool:
    return path.exists() and path.stat().st_size > 0


def _recent_manifest_row(
    *,
    label: str,
    product_type: str,
    year: int,
    month: int,
    days: list[str],
    ok: bool,
    error: str | None,
    path: Path,
) -> dict:
    return {
        "type": "recent",
        "label": label,
        "product_type": product_type,
        "year": year,
        "month": f"{month:02d}",
        "days": f"{days[0]}-{days[-1]}",
        "ok": ok,
        "error": error,
        "path": str(path),
    }


def download_last12(area, as_of: str, out_dir: Path, status_path: Path | None = None):
    """
    Download recent UTCI with month-level fallback:
      1) consolidated (preferred)
      2) intermediate only if consolidated is unavailable for that month

    Existing ZIPs are reused to avoid duplicate downloads.
    """
    dates = _dates_last12(as_of)
    man = []
    months = months_for_last_12(as_of)
    total = len(months)
    ok_count = 0
    fail_count = 0
    started_at = time.perf_counter()
    if status_path is not None:
        _write_progress_status(status_path, "recent_download", 0, total, ok_count, fail_count, started_at)

    for step, (y, m) in enumerate(months, start=1):
        days = _days_in_month_window(dates, y, m)
        if not days:
            continue

        con_zip = out_dir / f"recent_consolidated_{y}{m:02d}.zip"
        int_zip = out_dir / f"recent_intermediate_{y}{m:02d}.zip"

        # Prefer consolidated if already cached.
        if _zip_ok(con_zip):
            man.append(
                _recent_manifest_row(
                    label="consolidated",
                    product_type=PRODUCT_CONSOLIDATED,
                    year=y,
                    month=m,
                    days=days,
                    ok=True,
                    error=None,
                    path=con_zip,
                )
            )
            ok_count += 1
            if status_path is not None:
                _write_progress_status(
                    status_path,
                    "recent_download",
                    step,
                    total,
                    ok_count,
                    fail_count,
                    started_at,
                    current=f"{y}-{m:02d}",
                )
            print(
                f"Recent download progress {step}/{total} ({(step / total) * 100:.1f}%) month={y}-{m:02d} ok=True (cached consolidated)"
            )
            continue

        # Try consolidated download first.
        con_req = cds_request(
            area,
            years=[y],
            product_type=PRODUCT_CONSOLIDATED,
            months=[f"{m:02d}"],
            days=days,
        )
        con_ok, con_err = download_cds(DATASET_UTCI, con_req, con_zip)
        man.append(
            _recent_manifest_row(
                label="consolidated",
                product_type=PRODUCT_CONSOLIDATED,
                year=y,
                month=m,
                days=days,
                ok=con_ok,
                error=con_err,
                path=con_zip,
            )
        )
        if con_ok:
            ok_count += 1
            if status_path is not None:
                _write_progress_status(
                    status_path,
                    "recent_download",
                    step,
                    total,
                    ok_count,
                    fail_count,
                    started_at,
                    current=f"{y}-{m:02d}",
                )
            print(
                f"Recent download progress {step}/{total} ({(step / total) * 100:.1f}%) month={y}-{m:02d} ok=True (consolidated)"
            )
            continue

        # Consolidated unavailable: fallback to intermediate (cached or download).
        if _zip_ok(int_zip):
            man.append(
                _recent_manifest_row(
                    label="intermediate_cached_fallback",
                    product_type=PRODUCT_INTERIM,
                    year=y,
                    month=m,
                    days=days,
                    ok=True,
                    error=None,
                    path=int_zip,
                )
            )
            continue

        int_req = cds_request(
            area,
            years=[y],
            product_type=PRODUCT_INTERIM,
            months=[f"{m:02d}"],
            days=days,
        )
        int_ok, int_err = download_cds(DATASET_UTCI, int_req, int_zip)
        man.append(
            _recent_manifest_row(
                label="intermediate_fallback",
                product_type=PRODUCT_INTERIM,
                year=y,
                month=m,
                days=days,
                ok=int_ok,
                error=int_err,
                path=int_zip,
            )
        )

    return man


# ---- Extraction + manifests ----
def extract_zip_to_dir(zip_path: Path, out_dir: Path) -> list[Path]:
    """
    Extracts zip into out_dir/<zip_stem>/ and returns all .nc paths found after extraction.
    """
    out_subdir = ensure_dir(out_dir / zip_path.stem)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_subdir)
    return sorted(out_subdir.rglob("*.nc"))


def ensure_downloads(manifest: list[dict], kind: str) -> None:
    """
    Record manifest rows + raise if all downloads failed (helps catch auth/request issues early).
    """
    mf_path = DIRS["logs"] / f"cds_manifest_{kind}.csv"
    pd.DataFrame(manifest).to_csv(mf_path, index=False)
    update_run_metadata_artifact(f"cds_manifest_{kind}", mf_path, f"{len(manifest)} requests recorded")

    ok_count = sum(1 for r in manifest if r.get("ok"))
    if ok_count == 0:
        errs = [r.get("error") for r in manifest if r.get("error")]
        raise RuntimeError(
            f"All CDS downloads failed for '{kind}'. Example error: {errs[0] if errs else 'Unknown'}"
        )


def ensure_baseline_utci(
    area, start_year: int, end_year: int, out_dir: Path
) -> tuple[list[dict], list[Path]]:
    """
    Baseline is static. If extracted NetCDFs exist, do not download again.
    Guarded by AOI bounds hash (written to country cache) to prevent silent mismatch.
    """
    # Optional: early check that the cache hash exists / matches
    cached_hash2 = read_cached_bounds_hash(BOUNDS_HASH_PATH)
    if cached_hash2 and cached_hash2 != current_bounds_hash:
        raise RuntimeError(
            "Baseline cache AOI bounds hash mismatch (unexpected; should have been caught earlier).\n"
            f"Cached:  {cached_hash2}\n"
            f"Current: {current_bounds_hash}"
        )

    extracted_dir = ensure_dir(out_dir / "extracted")
    existing_nc = sorted(extracted_dir.rglob("*.nc"))
    if existing_nc:
        manifest = [
            {
                "type": "baseline",
                "product_type": "cached",
                "ok": True,
                "error": None,
                "path": "extracted_netcdf_cache",
            }
        ]
        ensure_downloads(manifest, "baseline")
        return manifest, sorted({p.resolve() for p in existing_nc})

    # Otherwise, download monthly consolidated zips then extract
    manifest = download_baseline_monthly(
        area,
        start_year,
        end_year,
        out_dir,
        product_type=PRODUCT_CONSOLIDATED,
        status_path=UTCI_DOWNLOAD_STATUS_BASELINE,
    )
    ensure_downloads(manifest, "baseline")

    nc_paths: list[Path] = []
    for row in manifest:
        zpath = Path(row["path"])
        if zpath.exists() and zpath.stat().st_size > 0:
            nc_paths.extend(extract_zip_to_dir(zpath, extracted_dir))

    nc_paths = sorted({p.resolve() for p in nc_paths})
    if not nc_paths:
        raise RuntimeError("Baseline ZIPs extracted but no .nc files found.")
    return manifest, nc_paths


def ensure_recent_utci(area, as_of: str, out_dir: Path) -> tuple[list[dict], list[Path]]:
    """
    Recent window downloads with per-month fallback to intermediate only when needed.
    """
    manifest = download_last12(area, as_of, out_dir, status_path=UTCI_DOWNLOAD_STATUS_RECENT)
    ensure_downloads(manifest, "recent")

    extracted_dir = ensure_dir(out_dir / "extracted")
    nc_paths: list[Path] = []
    for row in manifest:
        zpath = Path(row["path"])
        if zpath.exists() and zpath.stat().st_size > 0:
            nc_paths.extend(extract_zip_to_dir(zpath, extracted_dir))

    nc_paths = sorted({p.resolve() for p in nc_paths})
    if not nc_paths:
        raise RuntimeError("Recent ZIPs extracted but no .nc files found.")

    return manifest, nc_paths


# ---- Execute ensures using the authoritative CDS bbox from Cell 5 ----
if USE_RELATIVE_THRESHOLD:
    baseline_manifest, baseline_nc_paths = ensure_baseline_utci(
        cds_bbox, BASELINE_START_YEAR, BASELINE_END_YEAR, CACHE_DIRS["baseline_raw"]
    )
else:
    baseline_manifest, baseline_nc_paths = [], []
    print("Skipping baseline downloads (USE_RELATIVE_THRESHOLD=False).")

recent_manifest, recent_nc_paths = ensure_recent_utci(cds_bbox, AS_OF_DATE, DIRS["recent_raw"])

print(f"Baseline NC files: {len(baseline_nc_paths)}")
print(f"Recent NC files:   {len(recent_nc_paths)}")

# Record key outputs to metadata
run_metadata["cds_downloads"] = {
    "dataset": DATASET_UTCI,
    "relative_threshold_enabled": bool(USE_RELATIVE_THRESHOLD),
    "baseline_nc_count": len(baseline_nc_paths),
    "recent_nc_count": len(recent_nc_paths),
    "product_types": {
        "baseline": PRODUCT_CONSOLIDATED if USE_RELATIVE_THRESHOLD else None,
        "recent_primary": PRODUCT_CONSOLIDATED,
        "recent_fallback": PRODUCT_INTERIM,
        "recent_strategy": "per_month_consolidated_then_intermediate_fallback",
    },
    "aoi_bounds_hash": current_bounds_hash,
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 9 — Open NetCDFs and build daily-max UTCI DataArrays (recent always, baseline optional)


def _open_mfdataset(nc_paths: list[Path]) -> xr.Dataset:
    if not nc_paths:
        raise ValueError("No NetCDF paths provided to open.")
    # Use decode_times=True by default; CDS files should be CF-compliant
    ds = xr.open_mfdataset(
        [str(p) for p in nc_paths],
        combine="by_coords",
        parallel=False,
        chunks=XR_CHUNKS,
        decode_times=True,
        engine="netcdf4",
    )
    return ds


def _standardise_latlon(ds: xr.Dataset) -> xr.Dataset:
    # Common coordinate name variants
    rename = {}
    if "latitude" in ds.coords and "lat" not in ds.coords:
        rename["latitude"] = "lat"
    if "longitude" in ds.coords and "lon" not in ds.coords:
        rename["longitude"] = "lon"
    if rename:
        ds = ds.rename(rename)

    # Sometimes lat/lon are data_vars instead of coords
    for c in ["lat", "lon"]:
        if c in ds.data_vars and c not in ds.coords:
            ds = ds.set_coords(c)

    if "lat" not in ds.coords or "lon" not in ds.coords:
        raise ValueError(f"Could not find lat/lon coords. Coords present: {list(ds.coords)}")

    return ds


def _clip_to_bbox(ds: xr.Dataset, bounds_wsen: tuple[float, float, float, float]) -> xr.Dataset:
    # bounds in EPSG:4326
    minx, miny, maxx, maxy = bounds_wsen  # W, S, E, N

    # Expand by half a grid cell so edge-intersecting pixels are retained.
    lon_vals = ds["lon"].values
    lat_vals = ds["lat"].values
    dlon = float(np.abs(np.diff(lon_vals).mean())) if lon_vals.size > 1 else 0.0
    dlat = float(np.abs(np.diff(lat_vals).mean())) if lat_vals.size > 1 else 0.0

    minx_pad = minx - dlon / 2.0
    maxx_pad = maxx + dlon / 2.0
    miny_pad = miny - dlat / 2.0
    maxy_pad = maxy + dlat / 2.0

    ds = ds.where(
        (ds["lon"] >= minx_pad) & (ds["lon"] <= maxx_pad) & (ds["lat"] >= miny_pad) & (ds["lat"] <= maxy_pad),
        drop=True,
    )
    return ds


def _find_daily_max_utci(ds: xr.Dataset) -> xr.DataArray:
    """
    Robustly extract the daily maximum UTCI field from the CDS daily statistics files.

    We support two common structures:
      A) Separate variables for min/max/mean, e.g. 'utci_max' or similar
      B) A single variable with a 'statistic' (or similar) dimension containing 'maximum'
    """
    # Candidate data variables that look like UTCI
    vars_lower = {v.lower(): v for v in ds.data_vars}
    candidates = [vars_lower[k] for k in vars_lower if "utci" in k]

    if not candidates:
        raise ValueError(f"No data variables containing 'utci' found. Data vars: {list(ds.data_vars)}")

    # 1) Prefer explicit *_max variables
    max_like = [v for v in candidates if any(s in v.lower() for s in ["max", "maximum"])]
    if max_like:
        # Choose the most plausible (shortest name heuristic)
        v = sorted(max_like, key=len)[0]
        return ds[v]

    # 2) Otherwise, look for a statistic-like dimension
    # Pick a primary UTCI variable (prefer exact match if available)
    if "utci" in vars_lower:
        base_var = vars_lower["utci"]
    else:
        base_var = sorted(candidates, key=len)[0]

    da = ds[base_var]

    # Look for a statistic dimension and select 'maximum'
    stat_dims = [d for d in da.dims if d.lower() in ["statistic", "statistics", "stat", "quantile"]]
    if stat_dims:
        sd = stat_dims[0]
        # Try common labels
        coord = da[sd]
        # Convert to strings safely
        coord_str = coord.astype(str)
        # Common values: "maximum", "max", "daily_maximum", etc.
        # Select the first match containing 'max'
        matches = [i for i, s in enumerate(coord_str.values.tolist()) if "max" in s.lower()]
        if not matches:
            raise ValueError(f"Found statistic dim '{sd}' but no 'max' label in {coord_str.values}.")
        return da.isel({sd: matches[0]}).drop_vars(sd, errors="ignore")

    # 3) If neither case works, fail loudly with structure info
    raise ValueError(
        f"Could not determine daily max UTCI field. Candidates={candidates}; dims for '{base_var}'={da.dims}"
    )


def _to_celsius_if_needed(da: xr.DataArray) -> xr.DataArray:
    """
    Convert Kelvin->Celsius if values look like Kelvin.
    UTCI is often stored in K in CDS outputs.
    """
    # Sample a small subset to avoid computing the whole array
    sample = da.isel({da.dims[0]: 0}).mean().compute()
    v = float(sample.values)
    # Heuristic: if > 100, it's almost certainly Kelvin
    if v > 100:
        da = da - 273.15
        da.attrs["units"] = "C"
    return da


def _ensure_time_sorted_unique(da: xr.DataArray) -> xr.DataArray:
    # Sort time and drop duplicates (keep first)
    if "time" not in da.dims:
        raise ValueError("Expected a 'time' dimension in UTCI DataArray.")
    da = da.sortby("time")
    # Drop duplicate timestamps if present
    _, idx = np.unique(da["time"].values, return_index=True)
    da = da.isel(time=np.sort(idx))
    return da


def _filter_time_window(da: xr.DataArray, start: pd.Timestamp, end: pd.Timestamp) -> xr.DataArray:
    # Inclusive bounds
    return da.sel(time=slice(start, end))


# ---- Baseline: load only when relative threshold mode is enabled ----

utci_base = None
if USE_RELATIVE_THRESHOLD:
    if not baseline_nc_paths:
        raise RuntimeError(
            "Relative threshold mode requires baseline data, but no baseline NetCDF files were prepared."
        )
    ds_base = _open_mfdataset(baseline_nc_paths)
    ds_base = _standardise_latlon(ds_base)
    ds_base = _clip_to_bbox(ds_base, country_bounds_4326)

    utci_base = _find_daily_max_utci(ds_base)
    utci_base = _to_celsius_if_needed(utci_base)
    utci_base = _ensure_time_sorted_unique(utci_base)

    print(
        "Baseline time range:",
        str(utci_base["time"].min().values),
        "->",
        str(utci_base["time"].max().values),
    )

# ---- Recent: consolidated preferred; fill gaps with interim ----
# Partition recent_nc_paths based on their parent zip stem (we extracted into .../extracted/<zip_stem>/)
recent_con_nc = [p for p in recent_nc_paths if "recent_consolidated_" in str(p)]
recent_int_nc = [p for p in recent_nc_paths if "recent_intermediate_" in str(p)]

# Fallback if pattern matching fails (still works; will just treat all as consolidated)
if not recent_con_nc:
    recent_con_nc = recent_nc_paths
if not recent_int_nc:
    recent_int_nc = []

ds_recent_con = _open_mfdataset(recent_con_nc)
ds_recent_con = _standardise_latlon(ds_recent_con)
ds_recent_con = _clip_to_bbox(ds_recent_con, country_bounds_4326)
da_recent_con = _to_celsius_if_needed(_find_daily_max_utci(ds_recent_con))
da_recent_con = _ensure_time_sorted_unique(da_recent_con)

if recent_int_nc:
    ds_recent_int = _open_mfdataset(recent_int_nc)
    ds_recent_int = _standardise_latlon(ds_recent_int)
    ds_recent_int = _clip_to_bbox(ds_recent_int, country_bounds_4326)
    da_recent_int = _to_celsius_if_needed(_find_daily_max_utci(ds_recent_int))
    da_recent_int = _ensure_time_sorted_unique(da_recent_int)

    # Consolidated takes priority; use interim only where consolidated is missing
    utci_recent = da_recent_con.combine_first(da_recent_int)
else:
    utci_recent = da_recent_con

utci_recent = _filter_time_window(utci_recent, start_date, end_date)
utci_recent = _ensure_time_sorted_unique(utci_recent)

print(
    "Recent time range:",
    str(utci_recent["time"].min().values),
    "→",
    str(utci_recent["time"].max().values),
)
print("Recent n_days:", utci_recent.sizes.get("time"))

# ---- Persist quick structure summary for debugging ----
SUMMARY_PATH = DIRS["logs"] / "utci_open_summary.json"
summary = {
    "baseline": (
        {
            "enabled": True,
            "dims": dict(utci_base.sizes),
            "time_min": str(pd.to_datetime(utci_base["time"].min().values)),
            "time_max": str(pd.to_datetime(utci_base["time"].max().values)),
            "units": utci_base.attrs.get("units"),
        }
        if utci_base is not None
        else {"enabled": False}
    ),
    "recent": {
        "dims": dict(utci_recent.sizes),
        "time_min": str(pd.to_datetime(utci_recent["time"].min().values)),
        "time_max": str(pd.to_datetime(utci_recent["time"].max().values)),
        "units": utci_recent.attrs.get("units"),
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))
update_run_metadata_artifact(
    "utci_open_summary",
    SUMMARY_PATH,
    "Baseline/recent UTCI daily-max DataArray summary",
)

# Update run metadata
run_metadata["utci_processing"] = {
    "selected_field": "daily_max_utci_c",
    "relative_threshold_enabled": bool(USE_RELATIVE_THRESHOLD),
    "baseline_units": utci_base.attrs.get("units") if utci_base is not None else None,
    "recent_units": utci_recent.attrs.get("units"),
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 10 — Baseline reference statistics (optional; only for relative threshold mode)
# Output: CACHE_DIRS["baseline_ref"]/baseline_utci_ref_<ISO3>_<1991-2020>.json

import datetime as dt
from shapely.geometry import mapping

BASELINE_REF_PATH = (
    CACHE_DIRS["baseline_ref"] / f"baseline_utci_ref_{ISO3}_{BASELINE_START_YEAR}-{BASELINE_END_YEAR}.json"
)


def load_baseline_ref(path: Path) -> dict | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except Exception:
        return None


def save_baseline_ref(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2))


def clip_da_to_country_geom(da: xr.DataArray, geom_4326) -> xr.DataArray:
    """Clip a (time, lat, lon) DataArray to the country polygon using rioxarray."""
    da2 = da.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
    da2 = da2.rio.write_crs("EPSG:4326", inplace=False)
    return da2.rio.clip([mapping(geom_4326)], crs="EPSG:4326", drop=True, all_touched=True)


RELATIVE_UTCI_THRESHOLD_C = None

if USE_RELATIVE_THRESHOLD:
    baseline_ref = load_baseline_ref(BASELINE_REF_PATH)

    if baseline_ref is not None:
        cached_hash = baseline_ref.get("bounds_hash")
        if cached_hash and cached_hash != current_bounds_hash:
            raise RuntimeError(
                "Cached baseline reference stats exist but bounds_hash differs. "
                "Delete the country cache to force a rebuild."
            )
        p95_utci = float(baseline_ref["utci_p95_c"])
        print(f"Loaded cached baseline p95 UTCI: {p95_utci:.2f} C")
    else:
        if utci_base is None:
            raise RuntimeError("Relative threshold mode requires baseline UTCI data.")

        utci_base_country = clip_da_to_country_geom(utci_base, country_geom_4326)
        stacked = utci_base_country.stack(z=("time", "lat", "lon")).dropna("z")
        q = stacked.quantile([0.50, 0.95, 0.99], dim="z").compute()

        p50_utci = float(q.sel(quantile=0.50).values)
        p95_utci = float(q.sel(quantile=0.95).values)
        p99_utci = float(q.sel(quantile=0.99).values)

        baseline_ref = {
            "iso3": ISO3,
            "baseline_years": f"{BASELINE_START_YEAR}-{BASELINE_END_YEAR}",
            "bounds_wsen": [float(x) for x in country_bounds_4326],
            "bounds_hash": current_bounds_hash,
            "units": utci_base.attrs.get("units", "C"),
            "utci_p50_c": p50_utci,
            "utci_p95_c": p95_utci,
            "utci_p99_c": p99_utci,
            "computed_utc": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "source_dataset": DATASET_UTCI,
            "variable": "universal_thermal_climate_index_daily_statistics",
            "statistic_used": "daily_max",
            "mask": "admin2 union polygon (COD GDB), all_touched=True",
        }
        save_baseline_ref(BASELINE_REF_PATH, baseline_ref)
        update_run_metadata_artifact(
            "baseline_reference_stats",
            BASELINE_REF_PATH,
            "Cached baseline UTCI reference percentiles (country-level)",
        )
        print(f"Computed baseline p95 UTCI: {p95_utci:.2f} C")
        print(f"QC percentiles: p50={p50_utci:.2f} C, p99={p99_utci:.2f} C")

    RELATIVE_UTCI_THRESHOLD_C = p95_utci

run_metadata["baseline_reference"] = {
    "enabled": bool(USE_RELATIVE_THRESHOLD),
    "path": str(BASELINE_REF_PATH) if USE_RELATIVE_THRESHOLD else None,
    "bounds_hash": current_bounds_hash if USE_RELATIVE_THRESHOLD else None,
    "utci_p95_c": RELATIVE_UTCI_THRESHOLD_C,
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 11 — Extreme heat day masks + consecutive-day exposure (absolute thresholds by default)
# Outputs (per threshold):
#   - daily_exceedance_mask_<thr>.tif          (daily boolean mask: UTCImax > thr)
#   - exceedance_3day_mask_<thr>.tif           (boolean mask: any 3-consecutive-day run)
#   - pop_affected_mask_<thr>.tif              (population masked raster)
#   - qc_summary_<thr>.json                    (quick QC stats)

from shapely.geometry import mapping
import time
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject


def transform_from_latlon_centers(lat_vals: np.ndarray, lon_vals: np.ndarray):
    """Build an affine transform from 1D latitude/longitude center coordinates."""
    lat_vals = np.asarray(lat_vals, dtype="float64")
    lon_vals = np.asarray(lon_vals, dtype="float64")
    if lat_vals.size < 2 or lon_vals.size < 2:
        raise ValueError("Need at least 2 lat/lon points to derive resolution.")

    dlat = float(np.abs(np.diff(lat_vals).mean()))
    dlon = float(np.abs(np.diff(lon_vals).mean()))

    west = float(np.min(lon_vals) - dlon / 2.0)
    north = float(np.max(lat_vals) + dlat / 2.0)
    return rasterio.transform.from_origin(west, north, dlon, dlat)


# --- thresholds ---
THRESHOLDS = {f"abs_{int(t)}c": float(t) for t in ABS_THRESHOLDS_C}
if USE_RELATIVE_THRESHOLD:
    if RELATIVE_UTCI_THRESHOLD_C is None:
        raise RuntimeError("Relative threshold mode enabled, but RELATIVE_UTCI_THRESHOLD_C is missing.")
    THRESHOLDS["rel_p95"] = float(RELATIVE_UTCI_THRESHOLD_C)

K_CONSECUTIVE_DAYS = int(K_CONSEC_DAYS)

UTCI_RASTER_STATUS_PATH = DIRS["logs"] / "utci_raster_compute_status.json"
_raster_t0 = time.perf_counter()


def _write_utci_raster_status(processed: int, total: int, current: str | None = None):
    elapsed = time.perf_counter() - _raster_t0
    rate = processed / elapsed if elapsed > 0 else 0.0
    remaining = max(0, total - processed)
    eta = remaining / rate if rate > 0 else None
    payload = {
        "stage": "utci_raster_compute",
        "processed": int(processed),
        "total": int(total),
        "pct_complete": float((processed / total) * 100.0) if total else 100.0,
        "elapsed_seconds": float(round(elapsed, 2)),
        "rate_items_per_second": float(round(rate, 4)),
        "eta_seconds": None if eta is None else float(round(eta, 2)),
        "current": current,
    }
    UTCI_RASTER_STATUS_PATH.write_text(json.dumps(payload, indent=2))


# --- ensure output dirs ---
# --- output dirs (already defined in DIRS) ---
MASK_NATIVE_DIR = DIRS["masks_native"]
MASK_WORLDPOP_DIR = DIRS["masks_worldpop"]
POP_EXPOSED_DIR = DIRS["pop_exposed"]
QC_DIR = ensure_dir(DIRS["qc"] / "heat_masks")


def write_bool_geotiff(path: Path, mask2d: np.ndarray, ref_da: xr.DataArray) -> None:
    """
    Write a 2D boolean mask (lat, lon) as uint8 GeoTIFF aligned to ref_da.
    Assumes ref_da is EPSG:4326 with 1D lat/lon coordinates.
    """
    lat = ref_da["lat"].values
    lon = ref_da["lon"].values

    # Build transform from coordinate centers (not bounds) to avoid half-pixel shrink.
    height, width = mask2d.shape
    transform = transform_from_latlon_centers(lat, lon)

    path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs="EPSG:4326",
        transform=transform,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        nodata=255,
    ) as dst:
        dst.write(mask2d.astype("uint8"), 1)


def align_mask_to_worldpop(mask_da: xr.DataArray, worldpop_da: xr.DataArray) -> np.ndarray:
    """
    Reproject/resample a boolean mask in EPSG:4326 to exactly match the worldpop grid.
    Returns uint8 array in worldpop shape (1=exposed, 0=not).
    """
    # Source mask as uint8
    src = mask_da.astype("uint8").values

    # Define source transform from coordinate centers (EPSG:4326).
    lat = mask_da["lat"].values
    lon = mask_da["lon"].values
    src_transform = transform_from_latlon_centers(lat, lon)

    # Destination grid from worldpop
    dst_shape = worldpop_da.shape
    dst = np.zeros(dst_shape, dtype="uint8")

    # worldpop is typically (lat, lon) in its own CRS/transform via rioxarray
    dst_transform = worldpop_da.rio.transform()
    dst_crs = worldpop_da.rio.crs

    reproject(
        source=src,
        destination=dst,
        src_transform=src_transform,
        src_crs="EPSG:4326",
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
        src_nodata=255,
        dst_nodata=255,
    )
    return dst


def consecutive_k_exceedance(exceed_da: xr.DataArray, k: int) -> xr.DataArray:
    """
    Given exceed_da(time, lat, lon) boolean, return a 2D mask(lat, lon) indicating
    whether there exists any run of >=k consecutive True days.
    """
    # Ensure time is sorted
    exceed_da = exceed_da.sortby("time")

    # Rolling window sum over time; any window with sum==k implies k consecutive Trues
    roll = exceed_da.rolling(time=k).sum()

    # roll has NaNs for first k-1 days; treat as not exposed
    hit = (roll >= k).fillna(False)

    # Any day where a k-window ends yields exposure; reduce over time to 2D
    return hit.any(dim="time")


def clip_recent_to_country(da: xr.DataArray) -> xr.DataArray:
    """
    Clip recent UTCI to country polygon to avoid coastal/ocean cells contributing to masks.
    """
    da2 = da.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False)
    da2 = da2.rio.write_crs("EPSG:4326", inplace=False)
    return da2.rio.clip([mapping(country_geom_4326)], crs="EPSG:4326", drop=True, all_touched=True)


# Ensure recent UTCI is polygon-clipped for mask generation
utci_recent_country = clip_recent_to_country(utci_recent)

# Precompute a denominator: total population within the country
# (worldpop is already country-clipped per your workflow)
total_pop = float(worldpop.sum().compute().values)

print(f"Total population (WorldPop raster sum): {total_pop:,.0f}")

results_masks = {}
threshold_items = list(THRESHOLDS.items())
_write_utci_raster_status(0, len(threshold_items), None)

for idx, (thr_name, thr_c) in enumerate(threshold_items, start=1):
    print(f"\n--- Threshold: {thr_name} ({thr_c:.2f} °C) ---")

    # 1) Daily exceedance mask (time, lat, lon)
    exceed = utci_recent_country > thr_c

    # 2) Any-run-of-3-days mask (lat, lon)
    mask_3day = consecutive_k_exceedance(exceed, K_CONSECUTIVE_DAYS)

    # 3) Save masks as GeoTIFFs (in EPSG:4326 on the UTCI grid) for QC
    daily_exceed_path = MASK_NATIVE_DIR / f"any_exceed_{thr_name}.tif"
    mask_3day_path = MASK_NATIVE_DIR / f"exceed_{K_CONSECUTIVE_DAYS}day_{thr_name}.tif"

    # For daily exceed, write a quick “any exceed during window” 2D mask to keep file small
    any_exceed_2d = exceed.any(dim="time").fillna(False).values
    write_bool_geotiff(daily_exceed_path, any_exceed_2d, utci_recent_country.isel(time=0))

    mask_3day_2d = mask_3day.fillna(False).values
    write_bool_geotiff(mask_3day_path, mask_3day_2d, utci_recent_country.isel(time=0))

    update_run_metadata_artifact(
        f"mask_any_exceed_{thr_name}", daily_exceed_path, "Any-day exceedance mask (QC)"
    )
    update_run_metadata_artifact(
        f"mask_{K_CONSECUTIVE_DAYS}day_{thr_name}", mask_3day_path, ">=3 consecutive-day exceedance mask"
    )

    # 4) Align 3-day mask to WorldPop grid and compute affected population
    mask_worldpop = align_mask_to_worldpop(mask_3day, worldpop)
    mask_worldpop_path = MASK_WORLDPOP_DIR / f"exceed_{K_CONSECUTIVE_DAYS}day_{thr_name}_worldpop.tif"
    with rasterio.open(WORLDPOP_PATH, "r") as src:
        profile = src.profile.copy()
        profile.update(dtype="uint8", count=1, compress="deflate", predictor=2, nodata=255)
        with rasterio.open(mask_worldpop_path, "w", **profile) as dst:
            dst.write(mask_worldpop.astype("uint8"), 1)  # values are 0/1; 255 is reserved for NoData
    update_run_metadata_artifact(
        f"mask_{K_CONSECUTIVE_DAYS}day_{thr_name}_worldpop",
        mask_worldpop_path,
        ">=3-day exceedance mask (WorldPop grid)",
    )

    pop_masked = (worldpop.values * mask_worldpop).astype("float32")

    pop_affected = float(np.nansum(pop_masked))
    pct_affected = (pop_affected / total_pop * 100.0) if total_pop > 0 else np.nan

    # 5) Save population-masked raster (same grid as WorldPop)
    pop_mask_path = POP_EXPOSED_DIR / f"pop_affected_{thr_name}.tif"
    with rasterio.open(
        WORLDPOP_PATH,
        "r",
    ) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", count=1, compress="deflate", predictor=2, nodata=0)
        with rasterio.open(pop_mask_path, "w", **profile) as dst:
            dst.write(pop_masked, 1)

    update_run_metadata_artifact(
        f"pop_affected_{thr_name}", pop_mask_path, "Population masked by >=3-day exceedance"
    )

    # 6) QC summary
    qc = {
        "iso3": ISO3,
        "as_of_date": AS_OF_DATE,
        "threshold_name": thr_name,
        "threshold_c": float(thr_c),
        "k_consecutive_days": K_CONSECUTIVE_DAYS,
        "pop_total": total_pop,
        "pop_affected": pop_affected,
        "pct_affected": pct_affected,
        "mask_any_exceed_path": str(daily_exceed_path),
        "mask_3day_path": str(mask_3day_path),
        "pop_mask_path": str(pop_mask_path),
    }
    qc_path = QC_DIR / f"qc_summary_{thr_name}.json"
    qc_path.write_text(json.dumps(qc, indent=2))
    update_run_metadata_artifact(f"qc_summary_{thr_name}", qc_path, "QC summary for threshold")

    results_masks[thr_name] = qc
    _write_utci_raster_status(idx, len(threshold_items), thr_name)
    elapsed = time.perf_counter() - _raster_t0
    rate = idx / elapsed if elapsed > 0 else 0.0
    eta = (len(threshold_items) - idx) / rate if rate > 0 else float("nan")
    print(f"Population affected: {pop_affected:,.0f} ({pct_affected:.2f}%)")
    print(
        f"Raster progress {idx}/{len(threshold_items)} ({(idx / len(threshold_items)) * 100:.1f}%) threshold={thr_name} eta={eta:.1f}s"
    )

# Keep results in memory for next cells (admin2 aggregation)
threshold_results = results_masks

run_metadata["extreme_heat_masks"] = {
    "k_consecutive_days": K_CONSECUTIVE_DAYS,
    "thresholds": {k: float(v) for k, v in THRESHOLDS.items()},
    "results_summary": {
        k: {"pop_affected": v["pop_affected"], "pct_affected": v["pct_affected"]}
        for k, v in results_masks.items()
    },
    "progress_status_path": str(UTCI_RASTER_STATUS_PATH),
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 12 — Admin2 aggregation: % population exposed by threshold
# Produces a single CSV with one row per adm2_pcode and columns:
#   - pop_total
#   - pop_exposed_<thr>, pct_exposed_<thr> for each threshold in THRESHOLDS
#
# Uses WorldPop grid as the common raster grid.
# Inputs created in Cell 10:
#   - POP_EXPOSED_DIR / pop_affected_<thr>.tif   (population masked raster, WorldPop grid)
# WorldPop raster path: WORLDPOP_PATH (hardcoded per country)

import pandas as pd
import rasterio
from rasterio.features import rasterize

# ---- Load admin2 boundaries (already filtered to ISO3 earlier) ----
admin2 = admin2_gdf.copy()

if "adm2_pcode" not in admin2.columns:
    raise KeyError(f"'adm2_pcode' not found. Available columns: {list(admin2.columns)}")

admin2 = admin2[["adm2_pcode", "geometry"]].dropna(subset=["adm2_pcode", "geometry"]).reset_index(drop=True)

# ---- Open WorldPop as the reference grid ----
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float64")
    wp_transform = wp.transform
    wp_crs = wp.crs
    wp_shape = (wp.height, wp.width)
    wp_nodata = wp.nodata

print("WorldPop nodata:", wp_nodata)
print("WorldPop min/max:", np.nanmin(wp_arr), np.nanmax(wp_arr))
print("WorldPop negatives:", int((wp_arr < 0).sum()))

# Valid population pixels: finite, not nodata, and >= 0
pop_valid_mask = np.isfinite(wp_arr)
if wp_nodata is not None:
    pop_valid_mask &= wp_arr != float(wp_nodata)
pop_valid_mask &= wp_arr >= 0

# Reproject admin2 geometries to match WorldPop CRS
admin2_wp = admin2.to_crs(wp_crs)

# ---- Rasterise admin2 polygons to unique IDs (1..N) on the WorldPop grid ----
shapes = [(geom, i + 1) for i, geom in enumerate(admin2_wp.geometry)]
admin2_id_raster = rasterize(
    shapes=shapes,
    out_shape=wp_shape,
    transform=wp_transform,
    fill=0,
    dtype="int32",
    all_touched=True,
)

flat_ids = admin2_id_raster.ravel()
flat_pop = wp_arr.ravel()
flat_pop_ok = pop_valid_mask.ravel()

# Only include pixels that are inside an admin2 AND have valid population
valid_pop = (flat_ids > 0) & flat_pop_ok

ids_valid_pop = flat_ids[valid_pop]
pop_valid = flat_pop[valid_pop]

# Total pop by admin2 id
pop_total_by_id = np.bincount(ids_valid_pop, weights=pop_valid, minlength=len(admin2_wp) + 1)

# ---- Output table scaffold ----
out = pd.DataFrame(
    {
        "adm2_pcode": admin2_wp["adm2_pcode"].values,
        "admin2_id": np.arange(1, len(admin2_wp) + 1, dtype=int),
    }
)

out["pop_total"] = out["admin2_id"].map(
    lambda i: float(pop_total_by_id[i]) if i < len(pop_total_by_id) else 0.0
)

# QC: country total from admin2 sums (should be close to masked WorldPop sum)
country_pop_from_admin2 = float(out["pop_total"].sum())
country_pop_from_raster = float(np.nansum(wp_arr[pop_valid_mask]))
print(f"Country pop (sum admin2):  {country_pop_from_admin2:,.0f}")
print(f"Country pop (masked WP):  {country_pop_from_raster:,.0f}")

# ---- For each threshold, sum exposed population within admin2 ----
threshold_keys = list(THRESHOLDS.keys())

for thr_name in threshold_keys:
    # NOTE: In your Cell 10 you used pop_mask_path = POP_EXPOSED_DIR / f"pop_exposed_{thr_name}.tif"
    # but in your pasted Cell 11 you're using pop_affected_{thr_name}.tif.
    # Keep this consistent with whatever you actually wrote in Cell 10.
    pop_exposed_path = POP_EXPOSED_DIR / f"pop_affected_{thr_name}.tif"
    if not pop_exposed_path.exists():
        # fallback to the alternate filename used earlier in this refactor
        alt = POP_EXPOSED_DIR / f"pop_exposed_{thr_name}.tif"
        if alt.exists():
            pop_exposed_path = alt
        else:
            raise FileNotFoundError(
                f"Missing exposed population raster for {thr_name}: {pop_exposed_path} (or {alt})"
            )

    with rasterio.open(pop_exposed_path) as src:
        exp_arr = src.read(1).astype("float64")
        exp_nodata = src.nodata

        # Grid safety
        if exp_arr.shape != wp_shape:
            raise ValueError(f"Raster shape mismatch for {thr_name}: {exp_arr.shape} vs WorldPop {wp_shape}")
        if src.transform != wp_transform:
            raise ValueError(f"Raster transform mismatch for {thr_name} vs WorldPop.")
        if src.crs != wp_crs:
            raise ValueError(f"Raster CRS mismatch for {thr_name}: {src.crs} vs WorldPop {wp_crs}")

    # Exposed raster validity: finite, not nodata, >=0
    exp_ok = np.isfinite(exp_arr)
    if exp_nodata is not None:
        exp_ok &= exp_arr != float(exp_nodata)
    exp_ok &= exp_arr >= 0

    flat_exp = exp_arr.ravel()
    flat_exp_ok = exp_ok.ravel()

    # Only include pixels that are inside admin2, have valid population, AND valid exposed values
    valid_exp = valid_pop & flat_exp_ok

    ids_valid_exp = flat_ids[valid_exp]
    exp_valid = flat_exp[valid_exp]

    pop_exposed_by_id = np.bincount(ids_valid_exp, weights=exp_valid, minlength=len(admin2_wp) + 1)

    col_pop = f"pop_exposed_{thr_name}"
    col_pct = f"pct_exposed_{thr_name}"

    out[col_pop] = out["admin2_id"].map(
        lambda i: float(pop_exposed_by_id[i]) if i < len(pop_exposed_by_id) else 0.0
    )
    out[col_pct] = np.where(
        out["pop_total"] > 0,
        (out[col_pop] / out["pop_total"]) * 100.0,
        np.nan,
    )

# ---- Save table ----
OUT_CSV_PATH = DIRS["tables"] / f"{ISO3}_admin2_extreme_heat_exposure_{AS_OF_DATE}.csv"
out = out.drop(columns=["admin2_id"])
out.to_csv(OUT_CSV_PATH, index=False)

update_run_metadata_artifact(
    "admin2_extreme_heat_table",
    OUT_CSV_PATH,
    "Admin2 population % exposed (>=3 consecutive days; multiple UTCI thresholds)",
)

print("Wrote:", OUT_CSV_PATH)
print(out.head(10).to_string(index=False))

# Update run metadata
run_metadata["admin2_table"] = {
    "path": str(OUT_CSV_PATH),
    "n_admin2": int(len(out)),
    "thresholds": {k: float(v) for k, v in THRESHOLDS.items()},
    "k_consecutive_days": K_CONSECUTIVE_DAYS,
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 13 — QC checks for admin2 table + rasters
# Writes:
#   - DIRS["qc"]/qc_admin2_table_<AS_OF_DATE>.json
#   - DIRS["qc"]/qc_admin2_table_<AS_OF_DATE>.csv (flagged rows)

import numpy as np
import pandas as pd
import rasterio

QC_OUT_DIR = ensure_dir(DIRS["qc"] / "admin2_table_qc")

# ---- Load the admin2 exposure table we just wrote ----
df = pd.read_csv(OUT_CSV_PATH)

# Identify threshold columns
thr_names = list(THRESHOLDS.keys())
pop_cols = [f"pop_exposed_{t}" for t in thr_names]
pct_cols = [f"pct_exposed_{t}" for t in thr_names]

required_cols = ["adm2_pcode", "pop_total"] + pop_cols + pct_cols
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing expected columns in admin2 table: {missing}")

# ---- Basic table QC ----
qc = {}
qc["n_admin2"] = int(len(df))
qc["n_admin2_pop_total_na"] = int(df["pop_total"].isna().sum())
qc["n_admin2_pop_total_zero"] = int((df["pop_total"] == 0).sum())
qc["n_admin2_pop_total_negative"] = int((df["pop_total"] < 0).sum())

# % bounds checks
flags = pd.DataFrame({"adm2_pcode": df["adm2_pcode"].astype(str)})
flags["flag_pop_total_negative"] = df["pop_total"] < 0
flags["flag_pop_total_zero"] = df["pop_total"] == 0

for t in thr_names:
    pcol = f"pop_exposed_{t}"
    ccol = f"pct_exposed_{t}"

    flags[f"flag_{ccol}_na"] = df[ccol].isna()
    flags[f"flag_{ccol}_lt0"] = df[ccol] < 0
    flags[f"flag_{ccol}_gt100"] = df[ccol] > 100

    # exposed pop should not exceed total pop (allow tiny float tolerance)
    flags[f"flag_{pcol}_gt_total"] = (df[pcol] - df["pop_total"]) > 1e-6

qc["pct_minmax"] = {
    t: {
        "min": float(np.nanmin(df[f"pct_exposed_{t}"].values)),
        "max": float(np.nanmax(df[f"pct_exposed_{t}"].values)),
        "na": int(df[f"pct_exposed_{t}"].isna().sum()),
    }
    for t in thr_names
}

# Country totals from table
country_pop_total = float(df["pop_total"].sum(skipna=True))
qc["country_pop_total_from_table"] = country_pop_total

# Compare to WorldPop masked sum
with rasterio.open(WORLDPOP_PATH) as wp:
    wp_arr = wp.read(1).astype("float64")
    wp_nodata = wp.nodata
    pop_ok = np.isfinite(wp_arr)
    if wp_nodata is not None:
        pop_ok &= wp_arr != float(wp_nodata)
    pop_ok &= wp_arr >= 0
    country_pop_total_from_raster = float(np.nansum(wp_arr[pop_ok]))

qc["country_pop_total_from_worldpop_raster"] = country_pop_total_from_raster
qc["country_pop_total_diff_abs"] = float(country_pop_total - country_pop_total_from_raster)
qc["country_pop_total_diff_pct"] = (
    float((qc["country_pop_total_diff_abs"] / country_pop_total_from_raster) * 100.0)
    if country_pop_total_from_raster > 0
    else np.nan
)

# Raster existence + quick stats (for each threshold)
qc["raster_checks"] = {}
for t in thr_names:
    ras_path = POP_EXPOSED_DIR / f"pop_affected_{t}.tif"
    if not ras_path.exists():
        ras_path = POP_EXPOSED_DIR / f"pop_exposed_{t}.tif"  # fallback if you standardised differently

    if not ras_path.exists():
        qc["raster_checks"][t] = {"exists": False}
        continue

    with rasterio.open(ras_path) as src:
        arr = src.read(1).astype("float64")
        nod = src.nodata
        ok = np.isfinite(arr)
        if nod is not None:
            ok &= arr != float(nod)
        ok &= arr >= 0

        qc["raster_checks"][t] = {
            "exists": True,
            "path": str(ras_path),
            "crs": str(src.crs),
            "shape": [int(src.height), int(src.width)],
            "nodata": nod,
            "min": float(np.nanmin(arr[ok])) if ok.any() else np.nan,
            "max": float(np.nanmax(arr[ok])) if ok.any() else np.nan,
            "sum_exposed_pop": float(np.nansum(arr[ok])) if ok.any() else 0.0,
        }

# Summarise flagged rows
flag_cols = [c for c in flags.columns if c.startswith("flag_")]
flags["any_flag"] = flags[flag_cols].any(axis=1)
flagged = flags[flags["any_flag"]].copy()

# Write QC outputs
QC_JSON_PATH = QC_OUT_DIR / f"qc_admin2_table_{ISO3}_{AS_OF_DATE}.json"
QC_CSV_PATH = QC_OUT_DIR / f"qc_admin2_table_{ISO3}_{AS_OF_DATE}_flagged.csv"

QC_JSON_PATH.write_text(json.dumps(qc, indent=2))
flagged.to_csv(QC_CSV_PATH, index=False)

update_run_metadata_artifact("qc_admin2_table_json", QC_JSON_PATH, "QC summary for admin2 exposure outputs")
update_run_metadata_artifact("qc_admin2_table_flagged_rows", QC_CSV_PATH, "Flagged admin2 rows (QC)")

print("QC written:")
print(" -", QC_JSON_PATH)
print(" -", QC_CSV_PATH)
print(f"Flagged rows: {len(flagged)} / {len(df)}")
print(f"Country pop (table):  {country_pop_total:,.0f}")
print(f"Country pop (raster): {country_pop_total_from_raster:,.0f}")
print(f"Diff (%): {qc['country_pop_total_diff_pct']:.3f}%")

In [ ]:
# Cell 14 — Visualisations (QC figures) — memory-safe refactor
# Produces:
#   - exceedance_mask_worldpop_<thr>.png
#   - worldpop.png
#   - pop_affected_<thr>.png
#   - admin2_choropleth_pct_<thr>.png

import gc
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show as rioshow
from rasterio.enums import Resampling
from pathlib import Path

FIG_DIR = ensure_dir(DIRS["qc"] / "figures")

# Choose a threshold to plot (change as needed)
THR_TO_PLOT = "abs_38c"  # set to any key in THRESHOLDS, e.g. abs_32c, abs_38c, abs_46c, rel_p95

if THR_TO_PLOT not in THRESHOLDS:
    raise KeyError(f"THR_TO_PLOT='{THR_TO_PLOT}' not in THRESHOLDS. Available: {list(THRESHOLDS.keys())}")

# Paths produced earlier
mask_worldpop_path = DIRS["masks_worldpop"] / f"exceed_{K_CONSECUTIVE_DAYS}day_{THR_TO_PLOT}_worldpop.tif"
if not mask_worldpop_path.exists():
    print("Note: WorldPop-grid exceedance mask not found:", mask_worldpop_path)

pop_affected_path = POP_EXPOSED_DIR / f"pop_affected_{THR_TO_PLOT}.tif"
if not pop_affected_path.exists():
    pop_affected_path = POP_EXPOSED_DIR / f"pop_exposed_{THR_TO_PLOT}.tif"

if not pop_affected_path.exists():
    raise FileNotFoundError(f"Missing population-affected raster for THR_TO_PLOT: {pop_affected_path}")

# Load admin2 exposure table
df = pd.read_csv(OUT_CSV_PATH)

# Load admin2 geometries (filtered already)
admin2_plot = admin2_gdf.copy()
if "adm2_pcode" not in admin2_plot.columns:
    raise KeyError("Expected 'adm2_pcode' in admin2_gdf for choropleth.")
admin2_plot = admin2_plot[["adm2_pcode", "geometry"]].dropna().copy()

# Reproject admin2 boundaries to WorldPop CRS for raster overlays
admin2_wp = admin2_plot.to_crs(wp_crs)

# Join table to admin2
pct_col = f"pct_exposed_{THR_TO_PLOT}"
admin2_plot = admin2_plot.merge(df[["adm2_pcode", pct_col, "pop_total"]], on="adm2_pcode", how="left")


# ---- Helpers: memory-safe raster preview and plotting ----
def _read_raster_preview(
    src: rasterio.io.DatasetReader,
    is_mask: bool,
    max_dim: int = 1500,
) -> tuple[np.ndarray, rasterio.Affine, float | None]:
    """
    Read a downsampled raster preview so plotting never loads the full raster into RAM.
    Returns: (array float32, transform for preview, nodata)
    """
    # Choose scale factor so largest dimension <= max_dim
    scale = max(src.width / max_dim, src.height / max_dim, 1.0)
    out_w = int(src.width / scale)
    out_h = int(src.height / scale)

    # Resample: nearest for masks, average for continuous surfaces
    resampling = Resampling.nearest if is_mask else Resampling.average

    arr = src.read(
        1,
        out_shape=(out_h, out_w),
        resampling=resampling,
    ).astype("float32")

    # Correct transform for the resampled grid
    transform = src.transform * src.transform.scale(src.width / out_w, src.height / out_h)

    return arr, transform, src.nodata


def plot_raster_preview(
    path: Path,
    title: str,
    out_png: Path,
    is_mask: bool = False,
    overlay_admin2: bool = True,
    max_dim: int = 1500,
) -> None:
    """
    Plot a raster as a downsampled preview (max_dim controls the largest image dimension).
    This is safe for very large rasters (e.g. WorldPop 100m).
    """
    with rasterio.open(path) as src:
        arr, transform, nod = _read_raster_preview(src, is_mask=is_mask, max_dim=max_dim)

    # Minimal nodata/invalid handling without ballooning memory
    if nod is not None:
        arr = np.where(arr == float(nod), np.nan, arr)

    if not is_mask:
        # Population surfaces should be non-negative; treat negatives as invalid
        arr = np.where(arr < 0, np.nan, arr)

    fig, ax = plt.subplots()
    ax.set_title(title)
    rioshow(arr, transform=transform, ax=ax)

    if overlay_admin2:
        # Boundaries are small; safe to overlay
        admin2_wp.boundary.plot(ax=ax, linewidth=0.3, edgecolor="black")

    ax.set_axis_off()
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close(fig)

    # Explicit cleanup (helpful in long notebooks)
    del arr, fig, ax
    gc.collect()


# 1) Exceedance mask on WorldPop grid (if available)
if mask_worldpop_path.exists():
    plot_raster_preview(
        mask_worldpop_path,
        f"Extreme heat exceedance mask (1=exceedance, 0=no exceedance) (≥{K_CONSECUTIVE_DAYS} consecutive days) — {ISO3} — {THR_TO_PLOT}",
        FIG_DIR / f"exceedance_mask_worldpop_{ISO3}_{THR_TO_PLOT}.png",
        is_mask=True,
        overlay_admin2=True,
        max_dim=1800,  # masks look better a bit larger; still safe
    )

# 2) WorldPop (baseline population surface)
plot_raster_preview(
    Path(WORLDPOP_PATH),
    f"WorldPop (country-clipped) — {ISO3}",
    FIG_DIR / f"worldpop_{ISO3}.png",
    is_mask=False,
    overlay_admin2=True,
    max_dim=1800,
)

# 3) Population affected (WorldPop grid)
plot_raster_preview(
    pop_affected_path,
    f"Population affected (masked) — {ISO3} — {THR_TO_PLOT}",
    FIG_DIR / f"pop_affected_{ISO3}_{THR_TO_PLOT}.png",
    is_mask=False,
    overlay_admin2=True,
    max_dim=1800,
)

# 4) Admin2 choropleth (% exposed)
fig, ax = plt.subplots()
ax.set_title(f"Admin2 % population exposed — {ISO3} — {THR_TO_PLOT}")

admin2_plot.plot(
    column=pct_col,
    ax=ax,
    legend=True,
    linewidth=0.3,
    edgecolor="black",
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
ax.set_axis_off()
chor_png = FIG_DIR / f"admin2_choropleth_pct_{ISO3}_{THR_TO_PLOT}.png"
fig.savefig(chor_png, dpi=200, bbox_inches="tight")
plt.close(fig)
del fig, ax
gc.collect()

print("Figures written to:", FIG_DIR)
for p in sorted(FIG_DIR.glob(f"*{ISO3}*{THR_TO_PLOT}*.png")):
    print(" -", p)

# Record in metadata
run_metadata["figures"] = run_metadata.get("figures", {})
run_metadata["figures"][THR_TO_PLOT] = {
    "mask_worldpop": (
        str(FIG_DIR / f"exceedance_mask_worldpop_{ISO3}_{THR_TO_PLOT}.png")
        if mask_worldpop_path.exists()
        else None
    ),
    "worldpop": str(FIG_DIR / f"worldpop_{ISO3}.png"),
    "pop_affected": str(FIG_DIR / f"pop_affected_{ISO3}_{THR_TO_PLOT}.png"),
    "admin2_choropleth": str(chor_png),
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 15 — Export QC report (tables + figures) to a single PDF

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import json
import textwrap
from pathlib import Path

QC_PDF_DIR = ensure_dir(DIRS["qc"] / "reports")
QC_PDF_PATH = QC_PDF_DIR / f"QC_extreme_heat_{ISO3}_{AS_OF_DATE}.pdf"

# Load QC JSON written in Cell 12
QC_JSON_PATH = DIRS["qc"] / "admin2_table_qc" / f"qc_admin2_table_{ISO3}_{AS_OF_DATE}.json"
if not QC_JSON_PATH.exists():
    raise FileNotFoundError(f"QC JSON not found: {QC_JSON_PATH}")

qc = json.loads(QC_JSON_PATH.read_text())

# Collect figures written in Cell 13
FIG_DIR = DIRS["qc"] / "figures"
fig_paths = sorted(FIG_DIR.glob(f"*{ISO3}*.png"))

if not fig_paths:
    raise RuntimeError(f"No QC figures found in {FIG_DIR}")


# ---- Helper: text-only page ----
def add_text_page(pdf, title, lines):
    fig, ax = plt.subplots(figsize=(8.27, 11.69))  # A4 portrait
    ax.axis("off")

    y = 0.95
    ax.text(0.01, y, title, fontsize=14, fontweight="bold", va="top")
    y -= 0.04

    for line in lines:
        wrapped = textwrap.fill(line, width=110)
        ax.text(0.01, y, wrapped, fontsize=10, va="top")
        y -= 0.025 * (wrapped.count("\n") + 1)

        if y < 0.05:
            pdf.savefig(fig)
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(8.27, 11.69))
            ax.axis("off")
            y = 0.95

    pdf.savefig(fig)
    plt.close(fig)


with PdfPages(QC_PDF_PATH) as pdf:
    # ---- Cover / summary page ----
    summary_lines = [
        f"Country (ISO3): {ISO3}",
        f"Reference date (as-of): {AS_OF_DATE}",
        f"Consecutive-day definition: ≥{K_CONSECUTIVE_DAYS} days",
        "",
        "Thresholds evaluated:",
        *[f"  - {k}: {v}" for k, v in THRESHOLDS.items()],
        "",
        f"Number of admin2 units: {qc['n_admin2']}",
        "",
        "Population consistency check:",
        f"  - Country population (admin2 table): {qc['country_pop_total_from_table']:,.0f}",
        f"  - Country population (WorldPop raster): {qc['country_pop_total_from_worldpop_raster']:,.0f}",
        f"  - Absolute difference: {qc['country_pop_total_diff_abs']:,.0f}",
        f"  - Difference (%): {qc['country_pop_total_diff_pct']:.4f}%",
    ]

    add_text_page(
        pdf,
        title="QC Report — Extreme Heat Exposure (Admin2)",
        lines=summary_lines,
    )

    # ---- Percentage exposure stats ----
    pct_lines = ["Percentage exposed — summary by threshold:"]
    for t, stats in qc["pct_minmax"].items():
        pct_lines.extend(
            [
                f"{t}:",
                f"  - min: {stats['min']:.3f} %",
                f"  - max: {stats['max']:.3f} %",
                f"  - NA count: {stats['na']}",
            ]
        )

    add_text_page(
        pdf,
        title="Exposure Percentage — QC Summary",
        lines=pct_lines,
    )

    # ---- Raster checks ----
    raster_lines = ["Raster checks (population affected rasters):"]
    for t, info in qc["raster_checks"].items():
        raster_lines.append(f"{t}:")
        if not info.get("exists", False):
            raster_lines.append("  - Raster missing")
            continue

        raster_lines.extend(
            [
                f"  - Path: {info['path']}",
                f"  - CRS: {info['crs']}",
                f"  - Shape: {info['shape']}",
                f"  - NoData: {info['nodata']}",
                f"  - Min exposed pop: {info['min']}",
                f"  - Max exposed pop: {info['max']}",
                f"  - Sum exposed pop: {info['sum_exposed_pop']:,.0f}",
            ]
        )

    add_text_page(
        pdf,
        title="Raster QC",
        lines=raster_lines,
    )

    # ---- Figures ----
    for fig_path in fig_paths:
        fig = plt.figure(figsize=(8.27, 11.69))
        img = plt.imread(fig_path)
        plt.imshow(img)
        plt.axis("off")
        plt.title(fig_path.name, fontsize=10)
        pdf.savefig(fig)
        plt.close(fig)

# ---- Register artifact ----
update_run_metadata_artifact(
    "qc_report_pdf",
    QC_PDF_PATH,
    "QC report combining tables, checks, and figures",
)

print("QC PDF written to:")
print(QC_PDF_PATH)

run_metadata["qc_report"] = {
    "path": str(QC_PDF_PATH),
    "n_figures": len(fig_paths),
}
RUN_METADATA_PATH.write_text(json.dumps(run_metadata, indent=2))

In [ ]:
# Cell 16 - Run standardized post-run validation (schema + parity report)
import json
import subprocess
import sys
from pathlib import Path

run_dir = Path(DIRS["base"]).resolve()
cmd = [
    sys.executable,
    "scripts/utci_postrun.py",
    "--run-dir",
    str(run_dir),
]

result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    if result.stderr:
        print(result.stderr)
    raise RuntimeError("UTCI post-run checks failed.")

summary = json.loads(result.stdout)
print("Post-run status:", summary["status"])
print("Parity report:", summary["outputs"].get("parity_md"))